In [9]:
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

In [10]:
from src.prepare_splits import load_splits
from src.data_utils import get_Xy

from icecream import ic
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

In [11]:
splits = load_splits()
splits.keys()

dict_keys(['train', 'test'])

In [12]:
for split_name, data in splits.items():
    print(f'Split: {split_name}, n_samples: {len(data)}')
    print(data.columns)

Split: train, n_samples: 92
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion'],
      dtype='str')
Split: test, n_samples: 24
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion'],
      dtype='str')


In [13]:
def fit_linear_regression(splits=splits, x_cols=['tar', 'tbr'], y_col='hba1c', degree=1):  
    X_train, y_train, X_test, y_test = get_Xy(splits=splits, x_cols=x_cols, y_col=y_col, degree=degree)

    # Create and fit model
    model = LinearRegression()

    # Cross-validation on train set
    cv_scores = cross_validate(
        model, X_train, y_train,
        cv=5,  # 5-fold cross-validation
        scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
        return_train_score=True
    )

    print(f"Train R²: {cv_scores['train_r2'].mean():.4f} (+/- {cv_scores['train_r2'].std():.4f})")
    print(f"Val R²: {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})")
    print(f"Val RMSE: {np.sqrt(-cv_scores['test_neg_mean_squared_error'].mean()):.4f}")
    print(f"Val MAE: {-cv_scores['test_neg_mean_absolute_error'].mean():.4f}")

    # Fit on entire train set and evaluate on test set
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)

    print(f"\nTest R²: {r2_score(y_test, y_pred_test):.4f}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.4f}")
    print(f"Coefficients: tar={model.coef_[0]:.4f}, tbr={model.coef_[1]:.4f}")
    print(f"Intercept: {model.intercept_:.4f}")

    return model

In [14]:
fit_linear_regression(splits=splits, x_cols=['tar', 'tbr'], y_col='hba1c')

Train R²: 0.1779 (+/- 0.0082)
Val R²: 0.0573 (+/- 0.1063)
Val RMSE: 24.7280
Val MAE: 19.6854

Test R²: 0.1207
Test RMSE: 27.8954
Test MAE: 23.8258
Coefficients: tar=50.6948, tbr=-42.5692
Intercept: 64.3509


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[ 50.69,-42.57]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['tar','tbr']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,64.35
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


In [15]:
# Fit model with polynomial features
fit_linear_regression(splits=splits, x_cols=['tar', 'tbr'], y_col='hba1c', degree=2)

Creating degree 2 polynomial features ...
Original features: 2
Polynomial features: 5
Train R²: 0.2749 (+/- 0.0383)
Val R²: -0.0646 (+/- 0.4303)
Val RMSE: 25.3989
Val MAE: 19.6228

Test R²: 0.1050
Test RMSE: 28.1429
Test MAE: 23.8565
Coefficients: tar=171.7305, tbr=45.6018
Intercept: 52.2586


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ 171.73, 45.6 ,-170.33,-556.13, -87.75]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,52.26
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](5,)","[2.26,0.88,0.37,0.17,0.06]"


As shown in eda, tar and tbr alone are poor predictors of hba1c

In [16]:
fit_linear_regression(splits=splits, x_cols=['tar', 'tbr', 'age', 'gender', 'diabetes_type'], y_col='hba1c')

Train R²: 0.2014 (+/- 0.0110)
Val R²: -0.1229 (+/- 0.3136)
Val RMSE: 26.6671
Val MAE: 20.8357

Test R²: 0.0690
Test RMSE: 28.7031
Test MAE: 24.4618
Coefficients: tar=46.6561, tbr=-43.5924
Intercept: 53.6213


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ 46.66,-43.59, 0.2 , 2.79, -2.36]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['tar','tbr','age','gender','diabetes_type']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,53.62
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


In [18]:
fit_linear_regression(splits=splits, x_cols=['tar', 'tbr', 'age', 'gender', 'bmi',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion'], y_col='hba1c')

Train R²: 0.4534 (+/- 0.0384)
Val R²: -0.4493 (+/- 0.5408)
Val RMSE: 29.4548
Val MAE: 23.3931

Test R²: 0.1048
Test RMSE: 28.1461
Test MAE: 22.2464
Coefficients: tar=5.9638, tbr=-14.3421
Intercept: -50.0271


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](20,)","[ 5.96,-14.34, 0.2 ,..., -0.05, -0.14, -1.45]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](20,)","['tar','tbr','age',...,'n_hyper_events','rapid_changes','mean_excursion']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-50.03
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,20
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(19)
